# Extended Data Table 1

Environment: env 1

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from scipy.stats import mannwhitneyu
from scipy.stats import chi2_contingency


# load data

In [3]:
# initialize paths

dropbox_path = "C:\\Users\\ds7014\\NYU Langone Health Dropbox\\Daniel Sunko/Shenhav_lab/Visionary_AI"


full_dataset_path = os.path.join(dropbox_path,'code_data/Data/columbia')

cohort_pickles_path = os.path.join(full_dataset_path,'cohort_pickles')

# data_name = 'pec_hc'
data_name = 'features_all'
results_folder = os.path.join(full_dataset_path,data_name)

# pw_folder = os.path.join(full_dataset_path,'data_pop_wide')



control_file = os.path.join(cohort_pickles_path, 'final_hc_20251017.pkl')
clinical_file = os.path.join(full_dataset_path, 'clinical_features_all_20250930.csv')

n_comps = 10

case_file_sf = os.path.join(cohort_pickles_path, 'sf_case_list_20250910.pkl') 
case_file_nsf = os.path.join(cohort_pickles_path, 'nsf_case_list_20250910.pkl')
case_file_pec = os.path.join(cohort_pickles_path, 'final_pec_20251017.pkl')

case_file_eope = os.path.join(cohort_pickles_path, 'final_eope_20251017.pkl')
case_file_lope = os.path.join(cohort_pickles_path, 'final_lope_20251017.pkl')

In [4]:
# create binary column in metadata from column that hold list of diseases per subject
def create_disease_binary_columns(df, disease_column):
    df = df.copy()
    # Ensure the disease column is processed as a list if it's a string
    if df[disease_column].dtype == 'object':
        # Split string if diseases are comma-separated
        df[disease_column] = df[disease_column].str.replace(' ', '', regex=False).str.split(',').apply(
            lambda x: [i.strip() for i in x] if isinstance(x, list) else []
        )
    
    # Get unique diseases across all rows
    all_diseases = set()
    for diseases in df[disease_column]:
        if isinstance(diseases, list):
            all_diseases.update(diseases)
    
    # Create binary columns for each disease
    for disease in sorted(all_diseases):
        column_name = f'{disease.lower().replace(" ", "_")}'
        df[disease_column+'_'+column_name] = df[disease_column].apply(
            lambda x: 1 if disease in x else 0
        )
    
    return df

In [5]:
def prepare_clinical_file(md):
    md['Study ID'] = md['Study ID'].str.lower()
    # fill it in if there is a date
    md.loc[md['Date of Hypertension Diagnosis'].notna() & md['Diagnoses, Maternal, Hypertension'].isna(), 'Diagnoses, Maternal, Hypertension'] = 'temp_hypertension'
    md.loc[md['Diagnoses, Maternal, Hypertension, Preeclampsia'].notna() & md['Diagnoses, Maternal, Hypertension'].isna(), 'Diagnoses, Maternal, Hypertension'] = 'temp_hypertension'


    # create df with binary disease columns
    md_df = md[['Study ID','Medical Diagnoses','Cardiac Disease','Has the patient had any eye surgery in the past five years?','Diagnoses, Maternal','Diagnoses, Maternal, Hypertension','Diagnoses, Maternal, Gestational Diabetes',
                'Aspirin?','Statin?','BP meds?','Insulin?','Diagnoses, Maternal, Gestational Diabetes, A2','Tobacco Use']]

    md_df = md_df.rename(columns={'Medical Diagnoses':'med_diag','Cardiac Disease':'card','Has the patient had any eye surgery in the past five years?':'eye_surg','Diagnoses, Maternal':'diag_mat',
                        'Diagnoses, Maternal, Hypertension':'diag_mat_ht', 'Diagnoses, Maternal, Gestational Diabetes':'diag_mat_gd','Diagnoses, Maternal, Gestational Diabetes, A2':'diag_mat_gd_a2'})

    md_df = create_disease_binary_columns(md_df,'med_diag').drop('med_diag', axis=1).drop('med_diag_nonereported',axis=1)
    md_df = create_disease_binary_columns(md_df,'card').drop('card', axis=1)
    md_df = create_disease_binary_columns(md_df,'eye_surg').drop('eye_surg',axis=1).drop('eye_surg_none', axis=1)
    md_df = create_disease_binary_columns(md_df,'diag_mat').drop('diag_mat',axis=1)
    md_df = create_disease_binary_columns(md_df,'diag_mat_ht').drop('diag_mat_ht',axis=1)
    md_df = create_disease_binary_columns(md_df,'diag_mat_gd').drop('diag_mat_gd',axis=1)
    md_df = create_disease_binary_columns(md_df,'Aspirin?').drop('Aspirin?',axis=1).drop('Aspirin?_no',axis=1)
    md_df = create_disease_binary_columns(md_df,'Statin?').drop('Statin?',axis=1).drop('Statin?_no',axis=1)
    md_df = create_disease_binary_columns(md_df,'Insulin?').drop('Insulin?',axis=1).drop('Insulin?_no',axis=1)
    md_df = create_disease_binary_columns(md_df,'BP meds?').drop('BP meds?',axis=1).drop('BP meds?_no',axis=1)
    md_df = create_disease_binary_columns(md_df,'Tobacco Use').drop('Tobacco Use',axis=1)

    md_select = md.merge(md_df,on='Study ID')[['Study ID','Age at enrollment','Tobacco Use_currentsmoker','Tobacco Use_formersmoker','Past Pregnancies Diagnoses, Maternal, Hypertension','diag_mat_ivf','Maternal Ethnicity','Maternal Race',
    'Cardiac Disease','diag_mat_ama','med_diag_obesity','Diabetes Mellitus','Past Pregnancies Diagnoses, Maternal, Hypertension','Past Pregnancies Diagnoses, Maternal']]

    md_select = md_select.rename(columns={'Tobacco Use_currentsmoker':'Current Smoker','Tobacco Use_formersmoker':'Former Smoker',
                                        'diag_mat_ivf':'IVF','diag_mat_ama':'ama','med_diag_obesity':'obesity'})
    md_select = md_select.loc[:, ~md_select.columns.duplicated()]
    # display(md_select)
    # display(md_select['Past Preeclampsia'])
    md_select['Past Preeclampsia'] = md_select['Past Pregnancies Diagnoses, Maternal, Hypertension'].apply(lambda x: 1 if x=='Preeclampsia' else 0)
    md_select['Maternal hispanic'] = md_select['Maternal Ethnicity'].map({
        'Hispanic or Latinx': 1,
        'NOT Hispanic or Latinx': 0,
        'Unknown / Not Reported / Undefined / Declined': -1
    })
    md_select['Race_Grouped'] = md_select['Maternal Race'].replace({
        "Asian,Black or African American": "Multiracial",
        "Asian": "Asian",
        "White": "White",
        "Unknown / Not Reported / Undefined / Declined": "Unknown",
        "Black or African American": "Black",
        "Native Hawaiian or Other Pacific Islander": "Other",
        "American Indian/Alaska Native": "Other"
    })
    md_select = pd.get_dummies(md_select, columns=['Race_Grouped'], prefix='Maternal Race')
    md_select['Cardiac Hypertensive Disease'] = md_select['Cardiac Disease'].astype(str).str.contains('Hypertensive Disease', case=False, na=False).astype(int)
    md_select['ama_custom'] = (md_select['Age at enrollment'] > 35.).astype(int)
    md_select['Diabetes'] = md_select['Diabetes Mellitus'].isin(['Type I','Type II']).astype(int)
    md_select['Past Gestational HTN'] = md_select['Past Pregnancies Diagnoses, Maternal, Hypertension'].isin(['Gestational HTN']).astype(int)
    md_select['Past Gestational diabetes'] = md_select['Past Pregnancies Diagnoses, Maternal'].astype(str).str.contains('Gestational diabetes').astype(int)
    md_select['Past Hypertension'] = md_select['Past Pregnancies Diagnoses, Maternal'].astype(str).str.contains('Hypertension').astype(int)
    md_select = md_select.drop(['Maternal Ethnicity','Maternal Race','Cardiac Disease','Diabetes Mellitus','Past Pregnancies Diagnoses, Maternal, Hypertension','Past Pregnancies Diagnoses, Maternal'],axis=1)

    return(md_select)

In [6]:
def prepare_clinical_file_fmf(md):
    md['Study ID'] = md['Study ID'].str.lower()
    # fill it in if there is a date
    md.loc[md['Date of Hypertension Diagnosis'].notna() & md['Diagnoses, Maternal, Hypertension'].isna(), 'Diagnoses, Maternal, Hypertension'] = 'temp_hypertension'
    md.loc[md['Diagnoses, Maternal, Hypertension, Preeclampsia'].notna() & md['Diagnoses, Maternal, Hypertension'].isna(), 'Diagnoses, Maternal, Hypertension'] = 'temp_hypertension'


    # create df with binary disease columns
    md_df = md[['Study ID','Medical Diagnoses','Cardiac Disease','Has the patient had any eye surgery in the past five years?','Diagnoses, Maternal','Diagnoses, Maternal, Hypertension','Diagnoses, Maternal, Gestational Diabetes',
                'Aspirin?','Statin?','BP meds?','Insulin?','Diagnoses, Maternal, Gestational Diabetes, A2','Tobacco Use']]

    md_df = md_df.rename(columns={'Medical Diagnoses':'med_diag','Cardiac Disease':'card','Has the patient had any eye surgery in the past five years?':'eye_surg','Diagnoses, Maternal':'diag_mat',
                        'Diagnoses, Maternal, Hypertension':'diag_mat_ht', 'Diagnoses, Maternal, Gestational Diabetes':'diag_mat_gd','Diagnoses, Maternal, Gestational Diabetes, A2':'diag_mat_gd_a2'})

    md_df = create_disease_binary_columns(md_df,'med_diag').drop('med_diag', axis=1).drop('med_diag_nonereported',axis=1)
    md_df = create_disease_binary_columns(md_df,'card').drop('card', axis=1)
    md_df = create_disease_binary_columns(md_df,'eye_surg').drop('eye_surg',axis=1).drop('eye_surg_none', axis=1)
    md_df = create_disease_binary_columns(md_df,'diag_mat').drop('diag_mat',axis=1)
    md_df = create_disease_binary_columns(md_df,'diag_mat_ht').drop('diag_mat_ht',axis=1)
    md_df = create_disease_binary_columns(md_df,'diag_mat_gd').drop('diag_mat_gd',axis=1)
    md_df = create_disease_binary_columns(md_df,'Aspirin?').drop('Aspirin?',axis=1).drop('Aspirin?_no',axis=1)
    md_df = create_disease_binary_columns(md_df,'Statin?').drop('Statin?',axis=1).drop('Statin?_no',axis=1)
    md_df = create_disease_binary_columns(md_df,'Insulin?').drop('Insulin?',axis=1).drop('Insulin?_no',axis=1)
    md_df = create_disease_binary_columns(md_df,'BP meds?').drop('BP meds?',axis=1).drop('BP meds?_no',axis=1)
    md_df = create_disease_binary_columns(md_df,'Tobacco Use').drop('Tobacco Use',axis=1)

    md_select = md.merge(md_df,on='Study ID')[['Study ID','Age at enrollment','Tobacco Use_currentsmoker','Past Pregnancies Diagnoses, Maternal, Hypertension, Preeclampsia','diag_mat_ivf','Maternal Ethnicity','Maternal Race',
    'Cardiac Disease','Diabetes Mellitus','Past Pregnancies Diagnoses, Maternal, Hypertension','Past Pregnancies Diagnoses, Maternal','EDD by ultrasound','Term Births', 'Preterm Births','Insulin?']]

    md_select = md_select.rename(columns={'Tobacco Use_currentsmoker':'Current Smoker','Past Pregnancies Diagnoses, Maternal, Hypertension, Preeclampsia':'Past Preeclampsia','diag_mat_ivf':'IVF'})
    md_select['Past Preeclampsia'] = md_select['Past Preeclampsia'].apply(lambda x: 0 if x not in ['without severe features', 'with severe features'] else 1)
    md_select['Maternal hispanic'] = md_select['Maternal Ethnicity'].map({
        'Hispanic or Latinx': 1,
        'NOT Hispanic or Latinx': 0,
        'Unknown / Not Reported / Undefined / Declined': -1
    })
    md_select['Race_Grouped'] = md_select['Maternal Race'].replace({
        "Asian,Black or African American": "Multiracial",
        "Asian": "Asian",
        "White": "White",
        "Unknown / Not Reported / Undefined / Declined": "Unknown",
        "Black or African American": "Black",
        "Native Hawaiian or Other Pacific Islander": "Other",
        "American Indian/Alaska Native": "Other"
    })
    md_select = pd.get_dummies(md_select, columns=['Race_Grouped'], prefix='Maternal Race')
    md_select['Cardiac Hypertensive Disease'] = md_select['Cardiac Disease'].astype(str).str.contains('Hypertensive Disease', case=False, na=False).astype(int)
    md_select['Diabetes_type1'] = md_select['Diabetes Mellitus'].isin(['Type I']).astype(int)
    md_select['Diabetes_type2'] = md_select['Diabetes Mellitus'].isin(['Type II']).astype(int)
    md_select['Insulin?'] = md_select['Insulin?'].apply(lambda x: 1 if x=='Yes' else 0)
    md_select['Past Gestational diabetes'] = md_select['Past Pregnancies Diagnoses, Maternal'].astype(str).str.contains('Gestational diabetes').astype(int)
    md_select['Past Preterm Labor'] = md_select['Past Pregnancies Diagnoses, Maternal'].astype(str).str.contains('Preterm labor').astype(int)
    md_select['nulliparous'] = ((md_select['Term Births'] == 0) & (md_select['Preterm Births'] == 0)).astype(int)
    md_select = md_select.drop(['Maternal Ethnicity','Maternal Race','Cardiac Disease','Diabetes Mellitus','Past Pregnancies Diagnoses, Maternal, Hypertension','Past Pregnancies Diagnoses, Maternal','Term Births', 'Preterm Births'],axis=1)

    return(md_select)

In [7]:
md = pd.read_csv(os.path.join(dropbox_path,'AAAT2279 Liat 10/AAAT2279 Liat 10/MFMOphthBearellyWapn-DatasetOphthAAAT2279_DATA_LABELS_2025-08-01_1419.csv'))

In [8]:
clinical_feats_fmf = prepare_clinical_file_fmf(md)

In [9]:
clinical_feats = prepare_clinical_file(md)

In [10]:
severe_cases_ids = pd.read_pickle(case_file_sf)
severe_cases_ids = [x.lower() for x in severe_cases_ids]
severe_cases_ids = [x for x in severe_cases_ids if x!='cu0570'] # adjudicated as not pec
non_severe_cases_ids = pd.read_pickle(case_file_nsf)
non_severe_cases_ids = [x.lower() for x in non_severe_cases_ids]
eope_cases_ids = pd.read_pickle(case_file_eope)
eope_cases_ids = [x.lower() for x in eope_cases_ids]
lope_cases_ids = pd.read_pickle(case_file_lope)
lope_cases_ids = [x.lower() for x in lope_cases_ids]
clean_controls = pd.read_pickle(control_file)
clean_controls = [x.lower() for x in clean_controls]
covars_df = pd.read_csv(clinical_file, index_col=0)



pec_ids = pd.read_pickle(case_file_pec)
pec_ids = [x.lower() for x in pec_ids]

In [11]:
severe_cases_ids = [x for x in severe_cases_ids if x in pec_ids]
non_severe_cases_ids = [x for x in non_severe_cases_ids if x in pec_ids]

non_severe_cases_ids = list(set(non_severe_cases_ids+['cu0776']))
pec_ids = list(set(pec_ids+['cu0776']))
severe_cases_ids = list(set(severe_cases_ids))

eope_cases_ids = [x for x in eope_cases_ids if x in pec_ids]
lope_cases_ids = [x for x in pec_ids if x not in eope_cases_ids]

eope_cases_ids = list(set(eope_cases_ids))
lope_cases_ids = list(set(lope_cases_ids))

In [12]:
gdm_cases = pd.read_pickle(os.path.join(cohort_pickles_path,'gdm_cases_20250926.pkl'))
gdm_cases = [x.lower() for x in gdm_cases]
ght_cases = pd.read_pickle(os.path.join(cohort_pickles_path,'ght_cases_20250926.pkl'))
ght_cases = [x.lower() for x in ght_cases]
cht_cases = pd.read_pickle(os.path.join(cohort_pickles_path,'cht_cases_20250926.pkl'))
cht_cases = [x.lower() for x in cht_cases]

In [ ]:
pw1 = pd.read_pickle(os.path.join(cohort_pickles_path, 'final_hc_pw_1_controls_20251020.pkl'))
pw2 = pd.read_pickle(os.path.join(cohort_pickles_path, 'final_hc_pw_2_controls_20251020.pkl'))
pw3 = pd.read_pickle(os.path.join(cohort_pickles_path, 'final_hc_pw_3_controls_20251020.pkl'))
pw4 = pd.read_pickle(os.path.join(cohort_pickles_path, 'final_hc_pw_4_controls_20251020.pkl'))
pw5 = pd.read_pickle(os.path.join(cohort_pickles_path, 'final_hc_pw_5_controls_20251020.pkl'))
pw6 = pd.read_pickle(os.path.join(cohort_pickles_path, 'final_hc_pw_6_controls_20251020.pkl'))
pw7 = pd.read_pickle(os.path.join(cohort_pickles_path, 'final_hc_pw_7_controls_20251020.pkl'))
pw8 = pd.read_pickle(os.path.join(cohort_pickles_path, 'final_hc_pw_8_controls_20251020.pkl'))
pw9 = pd.read_pickle(os.path.join(cohort_pickles_path, 'final_hc_pw_9_controls_20251020.pkl'))
pw10 = pd.read_pickle(os.path.join(cohort_pickles_path, 'final_hc_pw_10_controls_20251020.pkl'))

pw_controls = list(set(pw1+pw2+pw3+pw4+pw5+pw6+pw7+pw8+pw9+pw10))
pw_controls = [x for x in pw_controls if x not in clean_controls]

In [14]:
pec_train = pd.read_pickle(os.path.join(cohort_pickles_path, 'pec_stratified_train_20251106.pkl'))
pec_test = pd.read_pickle(os.path.join(cohort_pickles_path, 'pec_strat_exp_test_20251113.pkl'))

In [15]:
pw_controls_combo = list(set(pw_controls+clean_controls))

In [16]:
cohort_ids = pw_controls_combo+pec_ids+ght_cases+cht_cases+gdm_cases
cohort_ids = list(set(cohort_ids))

cohort_full = clinical_feats_fmf['Study ID'].tolist()
new = [x for x in cohort_full if x not in cohort_ids]
cohort_ids = cohort_ids+new

### table 1 analysis

In [17]:
md_updated = pd.read_csv("C:\\Users\\ds7014\\NYU Langone Health Dropbox\\Daniel Sunko\\Shenhav_Lab\\Visionary_AI\\Updated_Cases_09_26_2025(All_subjects_09_2025).csv")
md_updated['Study ID'] = md_updated['Study ID'].str.lower()

md_updated = md_updated[md_updated['Study ID'].isin(cohort_ids)]

md = md_updated

In [18]:
covars_df = prepare_clinical_file(md)

In [19]:
all_card = md[md['Cardiac Disease'].str.lower().str.contains('hypertens', na=False)]['Study ID'].tolist()

In [20]:
card_with_pec = [x for x in all_card if x in pec_ids]

In [21]:
cht_cases_all = md[md['Cardiac Disease'].str.lower().str.contains('hypertens', na=False)]['Study ID'].tolist()
gdm_cases_all = md[md['Diagnoses, Maternal'].str.lower().str.contains('gestational diabetes', na=False)]['Study ID'].tolist()
dm_cases = md[md['Diabetes Mellitus'].isna()==False]['Study ID'].tolist()
endo_cases_all = md[md['Medical Diagnoses'].str.lower().str.contains('endocrine disease', na=False)]['Study ID'].tolist()

In [23]:
pw_stable_exc = pd.read_pickle("C:\\Users\\ds7014\\NYU Langone Health Dropbox\\Daniel Sunko\\Shenhav_Lab\\Visionary_AI\\code_data\\Data\\columbia\\cohort_pickles\\so_controls.pkl")

In [24]:


def get_vals(md, col, sets, get_sd=False, round_to=3, compare_to='pw_controls'):
    subj_means = {}
    subj_sds = {}
    subj_vals = {}

    for subj_set in sets.keys():
        subjects = sets[subj_set]
        subset = md[md['Study ID'].str.lower().isin(subjects)][col].dropna()
        mean_val = subset.mean()
        sd_val = subset.std()

        subj_means[subj_set] = mean_val
        subj_sds[subj_set] = sd_val
        subj_vals[subj_set] = subset

        if get_sd:
            print(f"{subj_set}: {mean_val:.{round_to}f} ± {sd_val:.{round_to}f}")
        else:
            print(f"{subj_set}: {mean_val:.{round_to}f}")


    # Compare each group against the pw_controls group
    for subj_set in sets.keys():
        if subj_set not in ['cohort', 'pw_controls', 'clean controls']:
            group_vals = subj_vals[subj_set]
            control_vals = subj_vals[compare_to]

            # Independent t-test (assumes unequal variances by default)
            t_stat, p_val = mannwhitneyu(group_vals, control_vals, nan_policy='omit')

            print(f"{subj_set} vs {compare_to}: t = {t_stat:.3f}, p = {p_val}")


def get_vals_count_card(md, col, sets, get_sd=False, compare_to='pw_controls'):
    subj_counts = {}
    subj_total = {}

    for subj_set, subjects in sets.items():
        subset = md[md['Study ID'].str.lower().isin(subjects)]
        subset = subset[subset['Cardiac Disease'].str.lower().str.contains(col, na=False)]
        count_val = len(subset)
        perc = (count_val / len(subjects)) * 100 if len(subjects) > 0 else 0
        print(f"{subj_set}: {count_val} ({perc:.1f}%)")

        subj_counts[subj_set] = count_val
        subj_total[subj_set] = len(subjects)

    for subj_set in sets.keys():
        if subj_set not in ['cohort','pw_controls','clean controls']:

            table = pd.DataFrame({
                "Has Disease": [subj_counts[subj_set], subj_counts[compare_to]],
                "No Disease": [subj_total[subj_set] - subj_counts[subj_set], subj_total[compare_to] - subj_counts[compare_to]]
            }, index=["PEC", "Controls"])
            chi2, p, dof, expected = chi2_contingency(table)
            print(f"{subj_set} vs {compare_to} Chi-square p-value = {p}")

def get_vals_past_pec(md, col, sets, get_sd=False, compare_to='pw_controls'):
    subj_counts = {}
    subj_total = {}
    for subj_set, subjects in sets.items():
        subset = md[md['Study ID'].str.lower().isin(subjects)]
        subset = subset[subset['Past Pregnancies Diagnoses, Maternal, Hypertension']=='Preeclampsia']
        count_val = len(subset)
        perc = (count_val / len(subjects)) * 100 if len(subjects) > 0 else 0
        print(f"{subj_set}: {count_val} ({perc:.1f}%)")

        subj_counts[subj_set] = count_val
        subj_total[subj_set] = len(subjects)

    for subj_set in sets.keys():
        if subj_set not in ['cohort','pw_controls','clean controls']:

            table = pd.DataFrame({
                "Has Disease": [subj_counts[subj_set], subj_counts[compare_to]],
                "No Disease": [subj_total[subj_set] - subj_counts[subj_set], subj_total[compare_to] - subj_counts[compare_to]]
            }, index=["PEC", "Controls"])
            chi2, p, dof, expected = chi2_contingency(table)
            print(f"{subj_set} vs {compare_to} Chi-square p-value = {p}")

def get_vals_count_smoker(md, col, sets, get_sd=False, compare_to='pw_controls'):
    subj_counts = {}
    subj_total = {}
    for subj_set, subjects in sets.items():
        subset = md[md['Study ID'].str.lower().isin(subjects)]
        subset = subset[(subset['Tobacco Use']=='Current Smoker') | (subset['Tobacco Use']=='Former Smoker')]
        count_val = len(subset)
        perc = (count_val / len(subjects)) * 100 if len(subjects) > 0 else 0
        print(f"{subj_set}: {count_val} ({perc:.1f}%)")

        subj_counts[subj_set] = count_val
        subj_total[subj_set] = len(subjects)

    for subj_set in sets.keys():
        if subj_set not in ['cohort','pw_controls','clean controls']:

            table = pd.DataFrame({
                "Has Disease": [subj_counts[subj_set], subj_counts[compare_to]],
                "No Disease": [subj_total[subj_set] - subj_counts[subj_set], subj_total[compare_to] - subj_counts[compare_to]]
            }, index=["PEC", "Controls"])
            chi2, p, dof, expected = chi2_contingency(table)
            print(f"{subj_set} vs {compare_to} Chi-square p-value = {p}")

def get_vals_count_ethnicity(md, col, sets, get_sd=False, compare_to='pw_controls'):
    subj_counts = {}
    subj_total = {}
    for subj_set, subjects in sets.items():
        subset = md[md['Study ID'].str.lower().isin(subjects)]
        subset = subset[subset['Maternal Ethnicity']=='Hispanic or Latinx']
        count_val = len(subset)
        perc = (count_val / len(subjects)) * 100 if len(subjects) > 0 else 0
        print(f"{subj_set}: {count_val} ({perc:.1f}%)")

        subj_counts[subj_set] = count_val
        subj_total[subj_set] = len(subjects)

    for subj_set in sets.keys():
        if subj_set not in ['cohort','pw_controls','clean controls']:

            table = pd.DataFrame({
                "Has Disease": [subj_counts[subj_set], subj_counts[compare_to]],
                "No Disease": [subj_total[subj_set] - subj_counts[subj_set], subj_total[compare_to] - subj_counts[compare_to]]
            }, index=["PEC", "Controls"])
            chi2, p, dof, expected = chi2_contingency(table)
            print(f"{subj_set} vs {compare_to} Chi-square p-value = {p}")


def get_vals_count_db(md, col, sets, get_sd=False, compare_to='pw_controls'):
    subj_counts = {}
    subj_total = {}
    for subj_set, subjects in sets.items():
        subset = md[md['Study ID'].str.lower().isin(subjects)]
        subset = subset[subset['Diabetes Mellitus'].isna()==False]
        count_val = len(subset)
        perc = (count_val / len(subjects)) * 100 if len(subjects) > 0 else 0
        print(f"{subj_set}: {count_val} ({perc:.1f}%)")

        subj_counts[subj_set] = count_val
        subj_total[subj_set] = len(subjects)

    for subj_set in sets.keys():
        if subj_set not in ['cohort','pw_controls','clean controls']:

            table = pd.DataFrame({
                "Has Disease": [subj_counts[subj_set], subj_counts[compare_to]],
                "No Disease": [subj_total[subj_set] - subj_counts[subj_set], subj_total[compare_to] - subj_counts[compare_to]]
            }, index=["PEC", "Controls"])
            chi2, p, dof, expected = chi2_contingency(table)
            print(f"{subj_set} vs {compare_to} Chi-square p-value = {p}")

def get_vals_count_diagmat(md, col, sets, get_sd=False, compare_to='pw_controls'):
    subj_counts = {}
    subj_total = {}
    for subj_set, subjects in sets.items():
        subset = md[md['Study ID'].str.lower().isin(subjects)]
        subset = subset[subset['Diagnoses, Maternal'].str.lower().str.contains(col, na=False)]
        count_val = len(subset)
        perc = (count_val / len(subjects)) * 100 if len(subjects) > 0 else 0
        print(f"{subj_set}: {count_val} ({perc:.1f}%)")

        subj_counts[subj_set] = count_val
        subj_total[subj_set] = len(subjects)

    for subj_set in sets.keys():
        if subj_set not in ['cohort','pw_controls','clean controls']:

            table = pd.DataFrame({
                "Has Disease": [subj_counts[subj_set], subj_counts[compare_to]],
                "No Disease": [subj_total[subj_set] - subj_counts[subj_set], subj_total[compare_to] - subj_counts[compare_to]]
            }, index=["PEC", "Controls"])
            chi2, p, dof, expected = chi2_contingency(table)
            print(f"{subj_set} vs {compare_to} Chi-square p-value = {p}")

def get_vals_count_meddiag(md, col, sets, get_sd=False, compare_to='pw_controls'):
    subj_counts = {}
    subj_total = {}
    for subj_set, subjects in sets.items():
        subset = md[md['Study ID'].str.lower().isin(subjects)]
        subset = subset[subset['Medical Diagnoses'].str.lower().str.contains(col, na=False)]
        count_val = len(subset)
        perc = (count_val / len(subjects)) * 100 if len(subjects) > 0 else 0
        print(f"{subj_set}: {count_val} ({perc:.1f}%)")

        subj_counts[subj_set] = count_val
        subj_total[subj_set] = len(subjects)

    for subj_set in sets.keys():
        if subj_set not in ['cohort','pw_controls','clean controls']:

            table = pd.DataFrame({
                "Has Disease": [subj_counts[subj_set], subj_counts[compare_to]],
                "No Disease": [subj_total[subj_set] - subj_counts[subj_set], subj_total[compare_to] - subj_counts[compare_to]]
            }, index=["PEC", "Controls"])
            chi2, p, dof, expected = chi2_contingency(table)
            print(f"{subj_set} vs {compare_to} Chi-square p-value = {p}")

def get_vals_count_np(md, col, sets, get_sd=False, compare_to='pw_controls'):
    subj_counts = {}
    subj_total = {}
    for subj_set, subjects in sets.items():
        subset = md[md['Study ID'].str.lower().isin(subjects)]
        subset = subset[subset['Living Children']==0]
        count_val = len(subset)
        perc = (count_val / len(subjects)) * 100 if len(subjects) > 0 else 0
        print(f"{subj_set}: {count_val} ({perc:.1f}%)")

        subj_counts[subj_set] = count_val
        subj_total[subj_set] = len(subjects)

    for subj_set in sets.keys():
        if subj_set not in ['cohort','pw_controls','clean controls']:

            table = pd.DataFrame({
                "Has Disease": [subj_counts[subj_set], subj_counts[compare_to]],
                "No Disease": [subj_total[subj_set] - subj_counts[subj_set], subj_total[compare_to] - subj_counts[compare_to]]
            }, index=["PEC", "Controls"])
            chi2, p, dof, expected = chi2_contingency(table)
            print(f"{subj_set} vs {compare_to} Chi-square p-value = {p}")

In [25]:
sets = {
    'cohort': cohort_ids,
    'pw_controls': pw_stable_exc, 
    'clean controls': clean_controls,
    'PEC': pec_ids,
    'severe': severe_cases_ids,
    'non_severe': non_severe_cases_ids,
    'eope':eope_cases_ids,
    'lope':lope_cases_ids,
    'ght':ght_cases}

# sets = {
#     'pec_train':pec_train,
#     'pec_test': pec_test
#     }


In [26]:
for k in sets.keys():
    print(f"{k}: {len(sets[k])}")

cohort: 1267
pw_controls: 1134
clean controls: 82
PEC: 55
severe: 38
non_severe: 17
eope: 20
lope: 35
ght: 49


In [27]:
get_vals_count_ethnicity(md, 'ethnicity', sets, )#compare_to='pec_test'

cohort: 1076 (84.9%)
pw_controls: 964 (85.0%)
clean controls: 73 (89.0%)
PEC: 46 (83.6%)
severe: 31 (81.6%)
non_severe: 15 (88.2%)
eope: 15 (75.0%)
lope: 31 (88.6%)
ght: 40 (81.6%)
PEC vs pw_controls Chi-square p-value = 0.9323284797054338
severe vs pw_controls Chi-square p-value = 0.7259367596663232
non_severe vs pw_controls Chi-square p-value = 0.9779103604988449
eope vs pw_controls Chi-square p-value = 0.35619961849573245
lope vs pw_controls Chi-square p-value = 0.732249638155738
ght vs pw_controls Chi-square p-value = 0.6584109317649431


In [28]:
get_vals_count_meddiag(md, 'endocrine', sets)#, compare_to='pec_test'

cohort: 113 (8.9%)
pw_controls: 95 (8.4%)
clean controls: 0 (0.0%)
PEC: 8 (14.5%)
severe: 6 (15.8%)
non_severe: 2 (11.8%)
eope: 5 (25.0%)
lope: 3 (8.6%)
ght: 9 (18.4%)
PEC vs pw_controls Chi-square p-value = 0.17936127635764662
severe vs pw_controls Chi-square p-value = 0.19096535551287158
non_severe vs pw_controls Chi-square p-value = 0.9527732087222321
eope vs pw_controls Chi-square p-value = 0.026520564983477456
lope vs pw_controls Chi-square p-value = 1.0
ght vs pw_controls Chi-square p-value = 0.03075551749892962


In [29]:
get_vals_count_meddiag(md, 'obesity', sets)#, compare_to='pec_test'

cohort: 445 (35.1%)
pw_controls: 392 (34.6%)
clean controls: 2 (2.4%)
PEC: 22 (40.0%)
severe: 15 (39.5%)
non_severe: 7 (41.2%)
eope: 9 (45.0%)
lope: 13 (37.1%)
ght: 32 (65.3%)
PEC vs pw_controls Chi-square p-value = 0.49591752983465354
severe vs pw_controls Chi-square p-value = 0.6515519409592418
non_severe vs pw_controls Chi-square p-value = 0.7553552951751745
eope vs pw_controls Chi-square p-value = 0.46271593154154356
lope vs pw_controls Chi-square p-value = 0.8926274049916129
ght vs pw_controls Chi-square p-value = 2.2255649480656423e-05


In [30]:
get_vals_count_diagmat(md, 'ivf', sets)#, compare_to='pec_test'

cohort: 17 (1.3%)
pw_controls: 11 (1.0%)
clean controls: 2 (2.4%)
PEC: 4 (7.3%)
severe: 4 (10.5%)
non_severe: 0 (0.0%)
eope: 1 (5.0%)
lope: 3 (8.6%)
ght: 0 (0.0%)
PEC vs pw_controls Chi-square p-value = 0.000517600367863999
severe vs pw_controls Chi-square p-value = 9.799849543421763e-06
non_severe vs pw_controls Chi-square p-value = 1.0
eope vs pw_controls Chi-square p-value = 0.5161037099630399
lope vs pw_controls Chi-square p-value = 0.0010272612193731657
ght vs pw_controls Chi-square p-value = 1.0


In [31]:
get_vals_count_np(md, 'livingchildren', sets)# #, compare_to='pec_test'

cohort: 484 (38.2%)
pw_controls: 423 (37.3%)
clean controls: 35 (42.7%)
PEC: 30 (54.5%)
severe: 18 (47.4%)
non_severe: 12 (70.6%)
eope: 10 (50.0%)
lope: 20 (57.1%)
ght: 25 (51.0%)
PEC vs pw_controls Chi-square p-value = 0.01511657963913496
severe vs pw_controls Chi-square p-value = 0.27579735868985256
non_severe vs pw_controls Chi-square p-value = 0.010540208236034427
eope vs pw_controls Chi-square p-value = 0.3525032564506102
lope vs pw_controls Chi-square p-value = 0.027366811482189304
ght vs pw_controls Chi-square p-value = 0.07378557626914675


In [32]:
get_vals_count_smoker(md, 'smok', sets)#, compare_to='pec_test'

cohort: 124 (9.8%)
pw_controls: 105 (9.3%)
clean controls: 3 (3.7%)
PEC: 9 (16.4%)
severe: 7 (18.4%)
non_severe: 2 (11.8%)
eope: 2 (10.0%)
lope: 7 (20.0%)
ght: 7 (14.3%)
PEC vs pw_controls Chi-square p-value = 0.13024189959636534
severe vs pw_controls Chi-square p-value = 0.10757972668275984
non_severe vs pw_controls Chi-square p-value = 1.0
eope vs pw_controls Chi-square p-value = 1.0
lope vs pw_controls Chi-square p-value = 0.0665339321568836
ght vs pw_controls Chi-square p-value = 0.3536793720044107


In [33]:
get_vals_count_db(md, 'db', sets)#, compare_to='pec_test'

cohort: 19 (1.5%)
pw_controls: 15 (1.3%)
clean controls: 0 (0.0%)
PEC: 3 (5.5%)
severe: 3 (7.9%)
non_severe: 0 (0.0%)
eope: 2 (10.0%)
lope: 1 (2.9%)
ght: 2 (4.1%)
PEC vs pw_controls Chi-square p-value = 0.059376997025161096
severe vs pw_controls Chi-square p-value = 0.010169338702144322
non_severe vs pw_controls Chi-square p-value = 1.0
eope vs pw_controls Chi-square p-value = 0.024017085000740866
lope vs pw_controls Chi-square p-value = 0.9753038732752439
ght vs pw_controls Chi-square p-value = 0.32919300792203204


In [34]:
get_vals_count_card(md, 'hypertens', sets)#, compare_to='pec_test'

cohort: 71 (5.6%)
pw_controls: 52 (4.6%)
clean controls: 0 (0.0%)
PEC: 10 (18.2%)
severe: 9 (23.7%)
non_severe: 1 (5.9%)
eope: 4 (20.0%)
lope: 6 (17.1%)
ght: 3 (6.1%)
PEC vs pw_controls Chi-square p-value = 3.807912045433682e-05
severe vs pw_controls Chi-square p-value = 1.2825463376860857e-06
non_severe vs pw_controls Chi-square p-value = 1.0
eope vs pw_controls Chi-square p-value = 0.007922828881453865
lope vs pw_controls Chi-square p-value = 0.0029356455077179278
ght vs pw_controls Chi-square p-value = 0.8777884077144725


In [35]:
get_vals_past_pec(md, 'Past Preeclampsia', sets)

cohort: 67 (5.3%)
pw_controls: 51 (4.5%)
clean controls: 1 (1.2%)
PEC: 9 (16.4%)
severe: 6 (15.8%)
non_severe: 3 (17.6%)
eope: 3 (15.0%)
lope: 6 (17.1%)
ght: 7 (14.3%)
PEC vs pw_controls Chi-square p-value = 0.00030524543535296667
severe vs pw_controls Chi-square p-value = 0.005112674308860291
non_severe vs pw_controls Chi-square p-value = 0.049158470821898614
eope vs pw_controls Chi-square p-value = 0.09480612144591462
lope vs pw_controls Chi-square p-value = 0.0025037784132149886
ght vs pw_controls Chi-square p-value = 0.005623698847399415


In [36]:
get_vals(md, 'Age at enrollment', sets, get_sd = True)

cohort: 29.907 ± 6.039
pw_controls: 29.807 ± 6.018
clean controls: 28.695 ± 5.734
PEC: 30.400 ± 6.873
severe: 31.868 ± 6.414
non_severe: 27.118 ± 6.909
eope: 32.150 ± 7.686
lope: 29.400 ± 6.260
ght: 30.550 ± 5.870
PEC vs pw_controls: t = 32537.500, p = 0.5862405082823927
severe vs pw_controls: t = 25422.000, p = 0.058677854265418126
non_severe vs pw_controls: t = 7115.500, p = 0.06333111868390841
eope vs pw_controls: t = 13400.500, p = 0.16272748254767655
lope vs pw_controls: t = 19137.000, p = 0.7187780907260666
ght vs pw_controls: t = 30083.000, p = 0.32547717266285126
